In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd

df = pd.read_excel("../data/raw/6S_AI_TASK-Loan_default_Loan_default.xlsx")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [3]:
id_col = 'LoanID'
target_col = 'Default'

num_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 
            'InterestRate', 'LoanTerm', 'DTIRatio']
cat_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 
            'LoanPurpose', 'HasCoSigner']

In [4]:
all_cols = set(num_cols + cat_cols + [id_col, target_col])

missing = set(df.columns) - all_cols
extra = all_cols - set(df.columns)

print("Missing from schema:", missing)
print("Extra in schema:", extra)

Missing from schema: set()
Extra in schema: set()


In [5]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # --- Ratio Features ---
    df['Loan_to_Income'] = df['LoanAmount'] / df['Income']
    df['Income_per_CreditLine'] = df['Income'] / df['NumCreditLines']
    df['Loan_per_CreditLine'] = df['LoanAmount'] / df['NumCreditLines']
    
    # --- Interaction Features ---
    df['Risk_Interaction'] = df['CreditScore'] * df['InterestRate']
    df['Debt_Stress'] = df['DTIRatio'] * df['LoanAmount']
    
    # --- Stability Feature ---
    df['Employment_Stability'] = df['MonthsEmployed'] / df['Age']
    
    # --- Binning ---
    df['CreditScore_bin'] = pd.cut(df['CreditScore'], bins=5)
    df['DTI_bin'] = pd.cut(df['DTIRatio'], bins=[0, 0.3, 0.6, 1])
    
    # --- Binary Encoding ---
    binary_map = {'Yes': 1, 'No': 0}
    
    df['HasMortgage'] = df['HasMortgage'].map(binary_map).astype(int)
    df['HasDependents'] = df['HasDependents'].map(binary_map).astype(int)
    df['HasCoSigner'] = df['HasCoSigner'].map(binary_map).astype(int)
    
    return df

In [6]:
df_eng = engineer_features(df)

df_eng.head()


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,...,HasCoSigner,Default,Loan_to_Income,Income_per_CreditLine,Loan_per_CreditLine,Risk_Interaction,Debt_Stress,Employment_Stability,CreditScore_bin,DTI_bin
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,...,1,0,0.588262,21498.500000,12646.750000,7919.60,22258.28,1.428571,"(519.6, 629.4]","(0.3, 0.6]"
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,...,1,0,2.467481,50432.000000,124440.000000,2202.98,84619.20,0.217391,"(409.8, 519.6]","(0.6, 1.0]"
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,...,0,1,1.534154,28069.333333,43062.666667,9547.67,40048.28,0.565217,"(409.8, 519.6]","(0.3, 0.6]"
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,...,0,0,1.412638,10571.000000,14933.000000,5253.01,10303.77,0.000000,"(739.2, 849.0]","(0.0, 0.3]"
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,...,0,0,0.447179,5109.250000,2284.750000,4120.83,6671.47,0.133333,"(629.4, 739.2]","(0.6, 1.0]"


In [7]:
df_eng.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default',
       'Loan_to_Income', 'Income_per_CreditLine', 'Loan_per_CreditLine',
       'Risk_Interaction', 'Debt_Stress', 'Employment_Stability',
       'CreditScore_bin', 'DTI_bin'],
      dtype='str')

In [8]:
df_eng.isnull().sum().sort_values(ascending=False)

LoanID                   0
Age                      0
Income                   0
LoanAmount               0
CreditScore              0
MonthsEmployed           0
NumCreditLines           0
InterestRate             0
LoanTerm                 0
DTIRatio                 0
Education                0
EmploymentType           0
MaritalStatus            0
HasMortgage              0
HasDependents            0
LoanPurpose              0
HasCoSigner              0
Default                  0
Loan_to_Income           0
Income_per_CreditLine    0
Loan_per_CreditLine      0
Risk_Interaction         0
Debt_Stress              0
Employment_Stability     0
CreditScore_bin          0
DTI_bin                  0
dtype: int64

In [9]:
num_cols_extended = [
    'Age', 'Income', 'LoanAmount', 'CreditScore',
    'MonthsEmployed', 'NumCreditLines',
    'InterestRate', 'LoanTerm', 'DTIRatio',
    
    # Engineered
    'Loan_to_Income', 'Income_per_CreditLine',
    'Loan_per_CreditLine', 'Risk_Interaction',
    'Debt_Stress', 'Employment_Stability'
]
cat_cols_extended = [
    'Education', 'EmploymentType', 'MaritalStatus',
    'LoanPurpose',
    
    # Binned features (important!)
    'CreditScore_bin', 'DTI_bin'
]
binary_cols = [
    'HasMortgage', 'HasDependents', 'HasCoSigner'
]

In [10]:
from  sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [11]:
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols_extended),
    ('cat', cat_pipeline, cat_cols_extended),
    ('bin', 'passthrough', binary_cols)
])

In [12]:
X = df_eng.drop(columns=[target_col, id_col])
y = df_eng[target_col]

X_transformed = preprocessor.fit_transform(X)

In [13]:
X_transformed.shape

(255246, 42)

In [14]:
import numpy as np
np.isnan(X_transformed).sum()

np.int64(0)

In [15]:
from sklearn.model_selection import train_test_split

X = df_eng.drop(columns=[target_col, id_col])
y = df_eng[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [16]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [17]:
from src.pipeline.preprocessing import DataPreprocessor

preprocessor = DataPreprocessor()

X_train_transformed = preprocessor.fit_transform(df)

ValueError: invalid literal for int() with base 10: 'Yes'

In [ ]:
# Saving the Preprocessor
import joblib

preprocessor = DataPreprocessor()
preprocessor.fit(pd.concat([X_train, y_train], axis=1))

joblib.dump(preprocessor, "../models/preprocessor.joblib")

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64')